# 문서 파싱과 스크래핑

HTML 구조에서 제목과 요약을 추출하고 robots.txt 규칙을 읽습니다.

## 실행 환경 준비

In [1]:
# 배포된 HTML 원문과 Ch 5 전달 파일의 경로를 준비합니다.
import json
from pathlib import Path
from urllib.robotparser import RobotFileParser

from bs4 import BeautifulSoup

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "labs/part1/ch04-data-tools/data/article.html").exists())
data_dir = root / "labs/part1/ch04-data-tools/data"
html = (data_dir / "article.html").read_text(encoding="utf-8")
handoff_path = data_dir / "article-text.json"
print("문서 길이:", len(html))

문서 길이: 270


## 기본 실행

In [2]:
# 기사 필드를 추출해 Ch 5가 읽을 JSON 파일로 보존합니다.
soup = BeautifulSoup(html, "lxml")
article = soup.select_one("article")
record = {"id":article["data-id"], "title":article.h1.get_text(strip=True),
          "summary":article.select_one(".summary").get_text(strip=True),
          "published":article.time["datetime"]}
record["text"] = f"{record['title']}\n{record['summary']}"
payload = {"schema_version":"ch04-parsed-article-v1",
           "source":"labs/part1/ch04-data-tools/data/article.html", "record":record}
handoff_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
print(record)
assert handoff == payload

{'id': 'A10', 'title': '골목상권 매출 회복', 'summary': '주말 방문객 증가로 음식점 매출이 늘었습니다.', 'published': '2026-09-01', 'text': '골목상권 매출 회복\n주말 방문객 증가로 음식점 매출이 늘었습니다.'}


## 중간 결과 관찰

In [3]:
# 화면 날짜 대신 기계 판독 속성이 반환되었는지 확인합니다.
assert record["id"] == "A10"
assert record["published"] == "2026-09-01"
print("제목:", record["title"])

제목: 골목상권 매출 회복


## 값 하나를 바꿔 비교

In [4]:
# robots 규칙은 크롤러 경로만 판단하며 이용 허가를 뜻하지 않습니다.
robots = RobotFileParser()
robots.parse(["User-agent: *", "Allow: /articles/", "Disallow: /private/"])
print("기사 경로 규칙:", robots.can_fetch("course-bot", "https://example.test/articles/A10"))
print("비공개 경로 규칙:", robots.can_fetch("course-bot", "https://example.test/private/A10"))

기사 경로 규칙: True
비공개 경로 규칙: False


## 실패 재현

존재하지 않는 선택자에 바로 `get_text`를 호출하면 `AttributeError`가 발생합니다.

In [5]:
# 없는 선택자 결과에 메서드를 호출하면 AttributeError가 발생합니다.
missing = soup.select_one(".subtitle")
try:
    missing.get_text(strip=True)
except AttributeError as error:
    print(type(error).__name__, "선택자 결과:", missing)
    assert missing is None

AttributeError 선택자 결과: None


## 수정

선택 결과를 확인하고 없는 필드는 `None`으로 반환합니다.

In [6]:
# 선택 결과를 확인하고 없는 선택 필드를 None으로 유지합니다.
subtitle_node = soup.select_one(".subtitle")
subtitle = subtitle_node.get_text(strip=True) if subtitle_node else None
fixed_record = {**record, "subtitle":subtitle}
print(fixed_record)
assert fixed_record["subtitle"] is None

{'id': 'A10', 'title': '골목상권 매출 회복', 'summary': '주말 방문객 증가로 음식점 매출이 늘었습니다.', 'published': '2026-09-01', 'text': '골목상권 매출 회복\n주말 방문객 증가로 음식점 매출이 늘었습니다.', 'subtitle': None}
